# 012_create_pipeline_run_log_table
Creates `ctrl.pipeline_run_log`: append-only run events (one `RUNNING` row and one final row
per run). Safe to re-run: the table is only created if it is missing.
`05_helper_scripts/052_pipeline_run_report` reads it.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## 1. ctrl.pipeline_run_log

In [ ]:
log_path = ctrl_table_path("pipeline_run_log")
if not is_table_folder(log_path):
    (
        literal_frame(None, RUN_LOG_SCHEMA)
        .write.format(TABLE_FORMAT)
        .mode("overwrite")
        .save(log_path.as_posix())
    )
register_table(CTRL_DATABASE, "pipeline_run_log", log_path)
print(f"{CTRL_DATABASE}.pipeline_run_log ready")